# Data Preparation

This notebook cleans and validates the datasets used in the Bay Area Transit Accessibility Analysis.

Data Sources:
- Bay Area Metro Vital Signs commute time data
- BART average weekday ridership data

Outputs:
- commute_times_clean.csv
- bart_stations_clean.csv

These cleaned datasets will later be loaded into a SQLite database for analysis.

## Commute Times Dataset
The dataset used in this analysis comes from the Bay Area Metro Vital Signs data portal, which provides regional indicators related to transportation, housing, and economic conditions. The specific dataset contains average commute times for Bay Area cities, broken down by transportation mode.

Each row represents a city-year combination and includes the following key columns:

jurisdiction – City or municipality where commuters reside

county – County associated with the city

year – Year the data was recorded

overall – Average commute time (minutes) across all transportation modes

drive_alone – Average commute time for commuters driving alone

carpool – Average commute time for commuters who carpool

transit – Average commute time for commuters using public transportation

walking – Average commute time for commuters who walk

source – Data source reference provided in the dataset

geometry – Geographic boundary data used for mapping and spatial analysis

Commute times are reported in minutes, allowing for direct comparison of travel burden across cities and transportation methods.

### Building the dataset

In [7]:
import numpy as np
import pandas as pd
# Load dataset and verify
transitdf = pd.read_csv('/workspaces/bay-area-commute-analysis/data/raw/Vital_Signs__Commute_Time_-_Cities_20260128.csv')
print(transitdf.head())

  jurisdiction   county   year    overall  drive_alone    carpool    transit  \
0      Alameda  Alameda  2,023  32.058017    28.568969  31.174674  49.872232   
1       Albany  Alameda  2,023  32.654894    28.664629  22.493036  50.307927   
2     Berkeley  Alameda  2,023  29.909062    28.897937  25.424362  48.796393   
3       Dublin  Alameda  2,023  37.806120    34.477289  35.424588  70.035601   
4   Emeryville  Alameda  2,023  30.503385    27.952522  29.770916  48.000000   

        walk                 source  \
0  23.343496  ACS_B08136_B08301_5YR   
1  13.220165  ACS_B08136_B08301_5YR   
2  15.245429  ACS_B08136_B08301_5YR   
3   8.469203  ACS_B08136_B08301_5YR   
4  18.687050  ACS_B08136_B08301_5YR   

                                            geometry  
0  MULTIPOLYGON (((-122.28286719299996 37.7637559...  
1  MULTIPOLYGON (((-122.28818215799998 37.8978400...  
2  MULTIPOLYGON (((-122.24692314999999 37.8853600...  
3  MULTIPOLYGON (((-121.85505706899994 37.7014150...  
4  MULTIP

### Data inspection

In [8]:
#Check column info, inpsect data types, check for missing values.
print(transitdf.dtypes)
#Commute times are numeric, however year is string datatype, will have to be converted to int. 
print('Null Data? ' + str(transitdf.isnull().values.any()))
#Missing data confirmed in some columns.
print('Same year? ' + str(transitdf['year'].nunique()==1))
#All data is confirmed from the same year.


jurisdiction        str
county              str
year                str
overall         float64
drive_alone     float64
carpool         float64
transit         float64
walk            float64
source              str
geometry            str
dtype: object
Null Data? True
Same year? True


### Data cleaning and preperation

In [9]:
#Convert year to int, not really important in this instance but if the dataset had different years, it would allow us to filter properly.
transitdf['year']=transitdf['year'].str.replace(',','').astype(int)
print(transitdf.info())
#41 cities in the dataset have no data for transit commute times, and will be removed.
transitdf=transitdf.dropna(subset=['transit'])
print("Rows:", len(transitdf))
print("Missing transit values:", transitdf['transit'].isnull().sum())
transitdf.to_csv('../data/processed/commute_times_clean.csv', index=False)

<class 'pandas.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   jurisdiction  101 non-null    str    
 1   county        101 non-null    str    
 2   year          101 non-null    int64  
 3   overall       101 non-null    float64
 4   drive_alone   63 non-null     float64
 5   carpool       62 non-null     float64
 6   transit       60 non-null     float64
 7   walk          63 non-null     float64
 8   source        101 non-null    str    
 9   geometry      101 non-null    str    
dtypes: float64(5), int64(1), str(4)
memory usage: 8.0 KB
None
Rows: 60
Missing transit values: 0


## Bart Ridership Dataset

The dataset in this analysis comes directly from the bart.gov website. The dataset contains average weekday ridership for each station in the Bay Area Rapid Transit system for January 2026.

Each entry represents a single station and includes the average number of riders entering or exiting that station on a typical weekday. The dataset also includes monthly and yearly ridership changes compared to previous periods. 

For the purpose of this analysis, the primary data we are interested in is the average weekday ridership for each station. This metric will be used to indentify stations with low ridership and explore whether those stations are located in areas with weaker public transit accessibility. 

It's important to note that the stations are not grouped by juristiction or city. Each station will have to be manually associated with their respective juristiction later on in the analysis so that ridership data can be properly compared with city level commute data.

### Building the dataset

In [10]:
ridershipdf=pd.read_csv('/workspaces/bay-area-commute-analysis/data/raw/bart_ridership_jan2026.csv')
ridershipdf.head()
ridershipdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   station             50 non-null     str  
 1   avg_weekday_riders  50 non-null     str  
 2   monthly_change      50 non-null     str  
 3   monthly_percent     50 non-null     str  
 4   yearly_change       50 non-null     str  
 5   yearly_percent      50 non-null     str  
dtypes: str(6)
memory usage: 2.9 KB


### Data Inspection

In [11]:
print(ridershipdf.dtypes)
#Ridership count and percentage change is stored as string, should be converted to int.
print('Null data? ' + str(ridershipdf.isnull().values.any()))
#Missing data in some columns.
print(ridershipdf.shape)
#60 rows despite only 50 existing bart stations. Should be rechecked after null values are removed.
print(ridershipdf.columns)
#Column names are correct
print('Duplicate rows:', ridershipdf.duplicated().sum())
#Nine duplicate rows.

station               str
avg_weekday_riders    str
monthly_change        str
monthly_percent       str
yearly_change         str
yearly_percent        str
dtype: object
Null data? True
(60, 6)
Index(['station', 'avg_weekday_riders', 'monthly_change', 'monthly_percent',
       'yearly_change', 'yearly_percent'],
      dtype='str')
Duplicate rows: 9


### Data Cleaning and Preparation

In [12]:
#Drop duplicate entries
ridershipdf=ridershipdf.drop_duplicates()
#Remove any rows with missing weekday rider data, as they can't be used for analysis
ridershipdf = ridershipdf.dropna(subset=['avg_weekday_riders'])
#Convert data types for analysis
ridershipdf['avg_weekday_riders']=ridershipdf['avg_weekday_riders'].str.replace(",","",regex=False).astype(int)
ridershipdf['monthly_percent']=ridershipdf['monthly_percent'].str.replace('%',"",regex=False).astype(float)
print(ridershipdf.shape)
print('Duplicate rows:', ridershipdf.duplicated().sum())
#50 stations, zero duplicate rows. The data is ready for analysis.
print("Rows:", len(ridershipdf))
print("Missing rider counts:", ridershipdf['avg_weekday_riders'].isnull().sum())
ridershipdf.to_csv('../data/processed/bart_stations_clean.csv',index=False)

(50, 6)
Duplicate rows: 0
Rows: 50
Missing rider counts: 0


## Data Preparation Summary

Commute Dataset
- Converted year to integer format
- Removed cities without transit commute data
- Exported cleaned commute dataset

BART Ridership Dataset
- Removed duplicate rows
- Removed records without ridership data
- Converted rider counts and percentages to numeric types
- Exported cleaned ridership dataset

The cleaned datasets are ready for database loading and analysis.